# DeltaSVM: GM12878 dsQTL Benchmark

Replicates the deltaSVM variant-effect benchmark from:

> Lee D, Gorkin DU, Baker M, et al. A method to predict the impact of regulatory variants from DNA sequence. *Nat Genet* 47(8):955–961 (2015). [doi:10.1038/ng.3331](https://doi.org/10.1038/ng.3331)

Using the **exact training sequences, negative sets, and test variants** published
by the authors ([Beer lab downloads](https://beerlab.org/deltasvm/)), and comparing
against their published 10-mer deltaSVM weight table.

Also evaluates in the context of:

> Pampari A, Shcherbina A, Kvon E, Kosicki M, et al. ChromBPNet: bias factorized,
> base-resolution deep learning models of chromatin accessibility reveal
> cis-regulatory sequence syntax, transcription factor footprints and regulatory
> variants. *bioRxiv* (2024). [doi:10.1101/2024.12.25.630221](https://doi.org/10.1101/2024.12.25.630221)

**Pipeline**: Download author data → train 5 gkm-SVMs (one per negative set) →
convert to deltaSVM weights → average → score dsQTL variants → evaluate
classification (AUROC, AUPRC) and regression (Spearman ρ) → compare to
published weights and ChromBPNet.

**Parameters**: l=10, k=6, d=3 (matching Lee 2015, NOT LS-GKM defaults of l=11, k=7, d=3).

### Requirements

```bash
pip install gkmsvm-lite[examples]
```

In [1]:
from pathlib import Path

import numpy as np

from gkmsvm.backend import HAS_CUPY, HAS_MLX, to_cpu

if HAS_CUPY:
    DEVICE = "cuda"
    print("NVIDIA GPU detected (CuPy) — using CUDA acceleration")
elif HAS_MLX:
    DEVICE = "mlx"
    print("Apple Silicon detected (MLX) — using Metal acceleration")
else:
    DEVICE = "cpu"
    print("No GPU detected — using CPU")

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

TARBALL_URL = "https://beerlab.org/deltasvm/downloads/gm12878_sequence_sets.tar.gz"
WEIGHTS_URL = "https://beerlab.org/deltasvm/downloads/SupplementaryTable_gm12878weights.txt"
GEO_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE31nnn/GSE31388/suppl/GSE31388_dsQtlTable.txt.gz"

KERNEL_L = 10
KERNEL_K = 6
KERNEL_D = 3
C_PARAM = 1.0
N_NEGSETS = 5

QUICK_TEST = False

NVIDIA GPU detected (CuPy) — using CUDA acceleration


## 1. Data Download

| Dataset | Source | Description |
|---------|--------|-------------|
| Training sequences | [Beer lab](https://beerlab.org/deltasvm/) | 22,384 GM12878 DNase peaks (300bp) + 5 GC-matched negative sets |
| dsQTL test variants | [Beer lab](https://beerlab.org/deltasvm/) | 574 dsQTLs + 27,735 controls (19bp, both alleles) |
| Published weights | [Beer lab](https://beerlab.org/deltasvm/) | 10-mer deltaSVM weight table (524,800 RC-collapsed entries) |
| Effect sizes | [GEO GSE31388](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE31388) | Degner et al. 2012 dsQTL association β coefficients |

In [2]:
import subprocess
import tarfile

TARBALL_PATH = DATA_DIR / "gm12878_sequence_sets.tar.gz"
SEQ_DIR = DATA_DIR / "gm12878_sequence_sets"
PUB_WEIGHTS = DATA_DIR / "SupplementaryTable_gm12878weights.txt"
GEO_TABLE = DATA_DIR / "GSE31388_dsQtlTable.txt.gz"

if QUICK_TEST:
    print("QUICK_TEST mode — using bundled Nanog fixtures")
else:
    if not SEQ_DIR.exists():
        if not TARBALL_PATH.exists():
            print("Downloading Beer lab sequence sets (14 MB)...")
            subprocess.run(["curl", "-sL", "-o", str(TARBALL_PATH), TARBALL_URL], check=True)
        print("Extracting...")
        with tarfile.open(TARBALL_PATH) as tar:
            tar.extractall(DATA_DIR)
        print(f"  → {SEQ_DIR}")
    else:
        print(f"Sequences: {SEQ_DIR}")

    if not PUB_WEIGHTS.exists():
        print("Downloading published deltaSVM weights (16 MB)...")
        subprocess.run(["curl", "-sL", "-o", str(PUB_WEIGHTS), WEIGHTS_URL], check=True)
        print(f"  → {PUB_WEIGHTS}")
    else:
        print(f"Published weights: {PUB_WEIGHTS}")

    if not GEO_TABLE.exists():
        print("Downloading GEO GSE31388 dsQTL effect sizes...")
        subprocess.run(["curl", "-sL", "-o", str(GEO_TABLE), GEO_URL], check=True)
        print(f"  → {GEO_TABLE}")
    else:
        print(f"Effect sizes: {GEO_TABLE}")

Sequences: data/gm12878_sequence_sets
Published weights: data/SupplementaryTable_gm12878weights.txt
Effect sizes: data/GSE31388_dsQtlTable.txt.gz


## 2. Load Training Data

Using the exact sequences published by Lee et al. 2015:
- **Positives**: 22,384 GM12878 DNase-seq peak regions (300bp)
- **Negatives**: 5 sets of GC-matched background sequences (22,384 each)

The paper trained 5 separate gkm-SVMs (one per negative set) and averaged
the resulting deltaSVM weight tables.

In [3]:
from gkmsvm import read_fasta, one_hot_encode, one_hot_decode

if QUICK_TEST:
    from pathlib import Path as _P
    fixtures = _P("../tests/fixtures")
    pos_seqs = [seq for _, seq in read_fasta(str(fixtures / "nanog_pos_train.fa.gz"))]
    neg_sets = [[seq for _, seq in read_fasta(str(fixtures / "nanog_neg_train.fa.gz"))]]
    N_NEGSETS = 1
    KERNEL_L, KERNEL_K, KERNEL_D = 8, 5, 3
    print(f"Quick test: {len(pos_seqs)} pos, 1 neg set, l={KERNEL_L} k={KERNEL_K} d={KERNEL_D}")
else:
    pos_seqs = [seq for _, seq in read_fasta(str(SEQ_DIR / "gm12878_shared.fa"))]
    SEQ_LEN = len(pos_seqs[0])
    print(f"Training positives: {len(pos_seqs):,} sequences, {SEQ_LEN}bp")

    neg_sets = []
    for i in range(1, N_NEGSETS + 1):
        negs = [seq for _, seq in read_fasta(str(SEQ_DIR / f"nullseqs_gm12878_shared.{i}.1.fa"))]
        neg_sets.append(negs)
        print(f"  Negative set {i}: {len(negs):,} sequences")

Training positives: 22,384 sequences, 300bp
  Negative set 1: 22,384 sequences
  Negative set 2: 22,384 sequences
  Negative set 3: 22,384 sequences
  Negative set 4: 22,384 sequences
  Negative set 5: 22,384 sequences


## 3. Train gkm-SVMs and Convert to DeltaSVM

Train 5 gkm-SVMs (one per negative set), convert each to a deltaSVM weight
table, and average the weights — matching the procedure from Lee et al. 2015.

In [4]:
from gkmsvm import train_gkmsvm
import time, copy

models = []
dsvms = []

for i, neg_seqs in enumerate(neg_sets):
    print(f"\n── Model {i+1}/{N_NEGSETS} ──")
    t0 = time.time()
    m = train_gkmsvm(
        pos_seqs, neg_seqs,
        kernel_type="estimated" if not QUICK_TEST else "direct",
        l=KERNEL_L, k=KERNEL_K, d=KERNEL_D,
        C=C_PARAM, solver="auto",
        device=DEVICE, verbose=True
    )
    print(f"  {m.num_support_vectors} SVs, {time.time() - t0:.1f}s")

    if DEVICE == "cuda":
        m.cuda()
    elif DEVICE == "mlx":
        m.mlx()

    t0 = time.time()
    d = m.to_deltasvm(device=DEVICE, verbose=True)
    print(f"  DeltaSVM conversion: {time.time() - t0:.1f}s")

    m.cpu()
    models.append(m)
    dsvms.append(d)

avg_weights = np.mean([to_cpu(d.weights) for d in dsvms], axis=0)
dsvm = copy.deepcopy(dsvms[0])
dsvm.cpu()
dsvm.weights = avg_weights

n_nonzero = int((avg_weights != 0).sum())
print(f"\nAveraged {len(dsvms)} models")
print(f"Weight table: {avg_weights.shape[0]:,} entries, {n_nonzero:,} non-zero")
print(f"Weight range: [{avg_weights.min():.4f}, {avg_weights.max():.4f}]")


── Model 1/5 ──
Moving training data to CUDA GPU
Gram matrix (14.9 GB) exceeds GPU memory — computing on GPU, solving on CPU
Computing 44768x44768 Gram matrix on GPU, storing on CPU...


Gram matrix: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1035/1035 [14:24<00:00,  1.20it/s]


Fitting SVC with sklearn...
Training complete: 21960 support vectors
  21960 SVs, 879.6s


to_deltasvm:   0%|                                                                                                                                                     | 0/11 [00:18<?, ?it/s]


OutOfMemoryError: Out of memory allocating 17,568,000,000 bytes (allocated so far: 18,020,604,928 bytes).

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 3))
    nonzero_w = avg_weights[avg_weights != 0]
    ax.hist(nonzero_w, bins=100, color="steelblue", alpha=0.8, edgecolor="none")
    ax.axvline(0, color="black", linewidth=0.5, linestyle="--")
    ax.set_xlabel(f"{KERNEL_L}-mer weight (averaged over {N_NEGSETS} models)")
    ax.set_ylabel("Count")
    ax.set_title(f"DeltaSVM weight distribution ({n_nonzero:,} non-zero / {avg_weights.shape[0]:,})")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("(install matplotlib for plots)")

## 4. Compare with Published Weight Table

Load the Lee 2015 published deltaSVM weight table and compare with our
retrained weights. High correlation validates our implementation.

In [ ]:
from gkmsvm import load_deltasvm_model

if not QUICK_TEST:
    pub_dsvm = load_deltasvm_model(str(PUB_WEIGHTS), l=KERNEL_L)
    pub_weights = to_cpu(pub_dsvm.weights)

    mask = (avg_weights != 0) | (pub_weights != 0)
    r = np.corrcoef(avg_weights[mask], pub_weights[mask])[0, 1]
    print(f"Weight correlation (non-zero entries): r = {r:.4f}")
    print(f"  Our non-zero: {(avg_weights != 0).sum():,}")
    print(f"  Published non-zero: {(pub_weights != 0).sum():,}")
else:
    print("Skipping weight comparison in QUICK_TEST mode")

In [ ]:
try:
    import matplotlib.pyplot as plt

    if not QUICK_TEST:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))

        ax = axes[0]
        mask = (avg_weights != 0) | (pub_weights != 0)
        ax.scatter(pub_weights[mask], avg_weights[mask], alpha=0.3, s=5,
                   color="steelblue", rasterized=True)
        lim = max(abs(pub_weights[mask]).max(), abs(avg_weights[mask]).max()) * 1.1
        ax.plot([-lim, lim], [-lim, lim], "k--", alpha=0.3)
        ax.set_xlabel("Published weight (Lee 2015)")
        ax.set_ylabel("Our weight (gkmsvm-lite)")
        ax.set_title(f"k-mer weight comparison (r={r:.4f})")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)

        ax = axes[1]
        residuals = avg_weights[mask] - pub_weights[mask]
        ax.hist(residuals, bins=100, color="coral", alpha=0.8, edgecolor="none")
        ax.axvline(0, color="black", linewidth=0.5, linestyle="--")
        ax.set_xlabel("Weight difference (ours − published)")
        ax.set_ylabel("Count")
        ax.set_title(f"Residuals (mean={residuals.mean():.4f}, std={residuals.std():.4f})")

        plt.tight_layout()
        plt.show()
except ImportError:
    pass

## 5. Score dsQTL Variants

dsQTLs (Degner et al. 2012) are genetic variants associated with changes in DNase I
hypersensitivity. The Beer lab tarball includes the exact 19bp test sequences
for all 28,309 variants (574 dsQTLs + 27,735 controls, both alleles).

In [ ]:
import pandas as pd

if QUICK_TEST:
    rng = np.random.default_rng(42)
    n_var = 200
    var_ref_seqs, var_alt_seqs, var_labels, var_effects = [], [], [], []
    published_scores = []
    all_seqs = pos_seqs + neg_sets[0]
    for i in range(n_var):
        seq = all_seqs[i % len(all_seqs)].upper()
        pos = rng.integers(10, len(seq) - 10)
        alt_base = rng.choice([b for b in "ACGT" if b != seq[pos]])
        var_ref_seqs.append(seq)
        var_alt_seqs.append(seq[:pos] + alt_base + seq[pos + 1:])
        is_sig = rng.random() < 0.1
        var_labels.append(1 if is_sig else 0)
        var_effects.append(rng.normal(0.5 if is_sig else 0.0, 0.3))
        published_scores.append(rng.normal(0, 3))
    var_labels = np.array(var_labels)
    var_effects = np.array(var_effects)
    published_scores = np.array(published_scores)
    print(f"Quick test: {n_var} synthetic variants, {var_labels.sum()} positive")
else:
    pos_major = [(n, s) for n, s in read_fasta(str(SEQ_DIR / "dsqtl_test_pos.major.fa"))]
    pos_minor = [(n, s) for n, s in read_fasta(str(SEQ_DIR / "dsqtl_test_pos.minor.fa"))]
    neg_major = [(n, s) for n, s in read_fasta(str(SEQ_DIR / "dsqtl_test_neg.major.fa"))]
    neg_minor = [(n, s) for n, s in read_fasta(str(SEQ_DIR / "dsqtl_test_neg.minor.fa"))]

    var_ref_seqs = [s for _, s in pos_major] + [s for _, s in neg_major]
    var_alt_seqs = [s for _, s in pos_minor] + [s for _, s in neg_minor]
    var_labels = np.array([1] * len(pos_major) + [0] * len(neg_major))

    print(f"dsQTL test set: {len(var_ref_seqs)} variants "
          f"({len(pos_major)} sig + {len(neg_major)} ctrl)")
    print(f"Variant sequence length: {len(var_ref_seqs[0])}bp")

    # Match GEO effect sizes via FASTA header coordinates
    # Headers are "chrom:start-end" (hg18); SNP is at center of 19bp window (start+9)
    geo = pd.read_csv(GEO_TABLE, sep="\t", compression="gzip")
    geo = geo.sort_values("Pr(>|t|)").drop_duplicates(subset=["Chr", "SNP"], keep="first")

    var_effects = np.full(len(var_ref_seqs), np.nan)
    for idx, (name, _) in enumerate(pos_major):
        parts = name.split(":")
        if len(parts) != 2:
            continue
        chrom = parts[0]
        coords = parts[1].split("-")
        if len(coords) != 2:
            continue
        try:
            snp_pos = int(coords[0]) + 9
            match = geo[(geo["Chr"] == chrom) & (geo["SNP"] == snp_pos)]
            if len(match) > 0:
                var_effects[idx] = float(match.iloc[0]["Estimate"])
        except ValueError:
            pass

    n_effects = (~np.isnan(var_effects)).sum()
    print(f"With GEO effect sizes: {n_effects}")

In [ ]:
X_var_ref = np.stack([one_hot_encode(s) for s in var_ref_seqs])
X_var_alt = np.stack([one_hot_encode(s) for s in var_alt_seqs])

t0 = time.time()
dsvm_var_deltas = to_cpu(dsvm.score_variants(X_var_ref, X_var_alt).flatten())
dsvm_time = time.time() - t0
print(f"DeltaSVM scoring: {len(var_ref_seqs)} variants in {dsvm_time:.2f}s")

if not QUICK_TEST:
    pub_deltas = to_cpu(pub_dsvm.score_variants(X_var_ref, X_var_alt).flatten())
    pub_corr = np.corrcoef(dsvm_var_deltas, pub_deltas)[0, 1]
    print(f"Our vs published deltaSVM scores: r = {pub_corr:.4f}")
    published_scores = pub_deltas

## 6. Classification Evaluation

Use |deltaSVM score| to predict which variants are significant dsQTLs.
We evaluate both our retrained model and the published Lee 2015 scores.

Published results (Pampari et al. 2024, Fig. 6):
- **gkm-SVM DeltaSVM AUPRC**: 0.19
- **ChromBPNet w/ ATAC (572M reads) AUPRC**: 0.54

In [ ]:
try:
    from sklearn.metrics import (
        roc_auc_score, roc_curve,
        average_precision_score, precision_recall_curve,
    )

    abs_dsvm = np.abs(dsvm_var_deltas)
    abs_published = np.abs(published_scores)
    baseline_rate = var_labels.mean()

    auroc = roc_auc_score(var_labels, abs_dsvm)
    auprc = average_precision_score(var_labels, abs_dsvm)
    auroc_pub = roc_auc_score(var_labels, abs_published)
    auprc_pub = average_precision_score(var_labels, abs_published)

    print(f"dsQTL Classification Results")
    print(f"{'─' * 50}")
    print(f"{''  :30s} {'AUROC':>8s}  {'AUPRC':>8s}")
    print(f"{'─' * 50}")
    print(f"{'gkmsvm-lite (retrained)':30s} {auroc:8.4f}  {auprc:8.4f}")
    print(f"{'Published (Lee 2015)':30s} {auroc_pub:8.4f}  {auprc_pub:8.4f}")
    print(f"{'─' * 50}")
    print(f"{'Baseline rate':30s} {'':>8s}  {baseline_rate:8.4f}")
    print(f"{'AUPRC / random':30s} {'':>8s}  {auprc / baseline_rate:8.2f}x")

    if not QUICK_TEST:
        print(f"\nPublished reference (Pampari et al. 2024, Fig. 6):")
        print(f"  gkm-SVM deltaSVM (DNase 68M):     AUPRC = 0.19")
        print(f"  Enformer (published):              AUPRC = 0.33")
        print(f"  ChromBPNet (DNase 68M):            AUPRC = 0.43")
        print(f"  ChromBPNet (ATAC 572M):            AUPRC = 0.54")

except ImportError:
    print("Install scikit-learn for AUROC/AUPRC: pip install scikit-learn")

In [ ]:
try:
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        roc_curve, roc_auc_score,
        precision_recall_curve, average_precision_score,
    )

    abs_dsvm = np.abs(dsvm_var_deltas)
    abs_published = np.abs(published_scores)
    baseline_rate = var_labels.mean()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    ax = axes[0]
    for scores, label, color in [
        (abs_dsvm, "gkmsvm-lite", "steelblue"),
        (abs_published, "Lee 2015", "coral"),
    ]:
        fpr, tpr, _ = roc_curve(var_labels, scores)
        auc = roc_auc_score(var_labels, scores)
        ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{label} ({auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve")
    ax.legend(loc="lower right", fontsize=9)

    ax = axes[1]
    for scores, label, color in [
        (abs_dsvm, "gkmsvm-lite", "steelblue"),
        (abs_published, "Lee 2015", "coral"),
    ]:
        prec, rec, _ = precision_recall_curve(var_labels, scores)
        ap = average_precision_score(var_labels, scores)
        ax.plot(rec, prec, color=color, linewidth=2, label=f"{label} (AP={ap:.3f})")
    ax.axhline(baseline_rate, color="k", linestyle="--", alpha=0.3, linewidth=1,
               label=f"Random ({baseline_rate:.3f})")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve")
    ax.legend(loc="upper right", fontsize=9)

    ax = axes[2]
    pos_sc = abs_dsvm[var_labels == 1]
    neg_sc = abs_dsvm[var_labels == 0]
    ax.hist(neg_sc, bins=50, alpha=0.6, label="Non-significant", color="gray", density=True)
    ax.hist(pos_sc, bins=50, alpha=0.6, label="dsQTL (sig.)", color="steelblue", density=True)
    ax.set_xlabel("|DeltaSVM Δscore|")
    ax.set_ylabel("Density")
    ax.set_title("Score Distribution")
    ax.legend()

    plt.tight_layout()
    plt.show()
except ImportError:
    pass

## 7. Regression Evaluation

Does the predicted Δscore correlate with the measured effect size from the
dsQTL association test? Effect sizes (β coefficients) come from GEO GSE31388
(Degner et al. 2012). Only significant dsQTLs have reliable effect sizes.

In [ ]:
from scipy.stats import spearmanr, pearsonr

has_effects = (~np.isnan(var_effects)).sum() >= 10

if has_effects:
    sig_mask = (var_labels == 1) & (~np.isnan(var_effects))
    sig_deltas = dsvm_var_deltas[sig_mask]
    sig_effects = var_effects[sig_mask]
    sig_published = published_scores[sig_mask]

    if len(sig_deltas) >= 10:
        rho, p_spearman = spearmanr(sig_deltas, sig_effects)
        rho_pub, p_pub = spearmanr(sig_published, sig_effects)
        r_val, p_pearson = pearsonr(sig_deltas, sig_effects)

        print(f"dsQTL Regression Results (significant dsQTLs with effect sizes)")
        print(f"{'─' * 55}")
        print(f"{''  :35s} {'Spearman ρ':>12s}  {'p-value':>10s}")
        print(f"{'─' * 55}")
        print(f"{'gkmsvm-lite (retrained)':35s} {rho:12.4f}  {p_spearman:10.2e}")
        print(f"{'Published (Lee 2015)':35s} {rho_pub:12.4f}  {p_pub:10.2e}")
        print(f"{'─' * 55}")
        print(f"N variants: {len(sig_deltas)}")
        print(f"Pearson r (gkmsvm-lite): {r_val:.4f}")
    else:
        print(f"Too few significant dsQTLs with effects ({sig_mask.sum()}) for regression")
else:
    print("No effect size data available — skipping regression evaluation")

In [ ]:
try:
    import matplotlib.pyplot as plt
    from scipy.stats import spearmanr

    has_effects = (~np.isnan(var_effects)).sum() >= 10

    if has_effects:
        sig_mask = (var_labels == 1) & (~np.isnan(var_effects))
        sig_deltas = dsvm_var_deltas[sig_mask]
        sig_effects = var_effects[sig_mask]
        sig_published = published_scores[sig_mask]

        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

        ax = axes[0]
        ax.scatter(sig_effects, sig_deltas, alpha=0.4, s=12, color="steelblue",
                   rasterized=True)
        rho, _ = spearmanr(sig_deltas, sig_effects)
        z = np.polyfit(sig_effects, sig_deltas, 1)
        x_line = np.linspace(sig_effects.min(), sig_effects.max(), 100)
        ax.plot(x_line, np.polyval(z, x_line), "r-", alpha=0.6, linewidth=1.5)
        ax.set_xlabel("Measured effect size (β)")
        ax.set_ylabel("gkmsvm-lite Δscore")
        ax.set_title(f"gkmsvm-lite (ρ={rho:.3f}, n={len(sig_deltas)})")
        ax.axhline(0, color="grey", linewidth=0.5, alpha=0.5)
        ax.axvline(0, color="grey", linewidth=0.5, alpha=0.5)

        ax = axes[1]
        ax.scatter(sig_effects, sig_published, alpha=0.4, s=12, color="coral",
                   rasterized=True)
        rho_pub, _ = spearmanr(sig_published, sig_effects)
        z = np.polyfit(sig_effects, sig_published, 1)
        ax.plot(x_line, np.polyval(z, x_line), "r-", alpha=0.6, linewidth=1.5)
        ax.set_xlabel("Measured effect size (β)")
        ax.set_ylabel("Published Δscore (Lee 2015)")
        ax.set_title(f"Published (ρ={rho_pub:.3f}, n={len(sig_published)})")
        ax.axhline(0, color="grey", linewidth=0.5, alpha=0.5)
        ax.axvline(0, color="grey", linewidth=0.5, alpha=0.5)

        ax = axes[2]
        ax.scatter(published_scores, dsvm_var_deltas, alpha=0.1, s=3, color="gray",
                   rasterized=True)
        ax.scatter(sig_published, sig_deltas, alpha=0.5, s=12, color="steelblue",
                   label="Significant dsQTLs", rasterized=True)
        r_vs_pub = np.corrcoef(dsvm_var_deltas, published_scores)[0, 1]
        ax.set_xlabel("Published Δscore (Lee 2015)")
        ax.set_ylabel("gkmsvm-lite Δscore")
        ax.set_title(f"Score correlation (r={r_vs_pub:.3f}, n={len(dsvm_var_deltas)})")
        lo = min(published_scores.min(), dsvm_var_deltas.min())
        hi = max(published_scores.max(), dsvm_var_deltas.max())
        ax.plot([lo, hi], [lo, hi], "k--", alpha=0.3)
        ax.legend(fontsize=8)

        plt.tight_layout()
        plt.show()
except ImportError:
    pass

## 8. Comparison with Published Results

Summary of dsQTL benchmark results from the literature and this notebook.
AUPRC values from Pampari et al. 2024, Figure 6.

| Method | AUPRC | Source |
|--------|-------|--------|
| gkm-SVM deltaSVM (DNase 68M) | 0.19 | Pampari et al. 2024 |
| Enformer (published) | 0.33 | Pampari et al. 2024 |
| ChromBPNet (DNase 68M) | 0.43 | Pampari et al. 2024 |
| Enformer (revised local) | 0.53 | Pampari et al. 2024 |
| ChromBPNet (ATAC 572M) | 0.54 | Pampari et al. 2024 |
| **gkmsvm-lite deltaSVM** | *see above* | This notebook |
| **Published weight repro** | *see above* | Lee 2015 weights |

ChromBPNet improves over deltaSVM by modeling nonlinear sequence
dependencies and base-resolution accessibility profiles within its
~2 kb input window. DeltaSVM's l-mer linear approximation misses
these interactions but remains a fast, interpretable baseline that
scores variants in seconds.

## 9. CLI Equivalents

The same pipeline using the `gkmsvm` command-line interface:

In [ ]:
print("""\
# Train on DNase-seq peaks (repeat for each negative set)
gkmsvm -v train \\
    -p gm12878_shared.fa -n nullseqs_gm12878_shared.1.1.fa \\
    -o gm12878_model1.npz \\
    -t estimated -l 10 -k 6 -d 3 \\
    -C 1.0 --solver auto

# Convert to DeltaSVM weights
gkmsvm -v to-deltasvm \\
    -m gm12878_model1.npz \\
    -o gm12878_deltasvm_1.tsv

# Score variants with DeltaSVM (fast, linear)
gkmsvm deltasvm \\
    -w gm12878_deltasvm_avg.tsv \\
    -i variants.fa -l 10 \\
    -o dsvm_scores.tsv

# Score variants with full SVM (slower, exact)
gkmsvm score-variants \\
    -m gm12878_model1.npz \\
    --ref ref.fa --alt alt.fa \\
    -o svm_deltas.tsv
""")

## References

1. Lee D, Gorkin DU, Baker M, et al. A method to predict the impact of regulatory variants from DNA sequence. *Nat Genet* 47(8):955–961 (2015). [doi:10.1038/ng.3331](https://doi.org/10.1038/ng.3331)
2. Degner JF, Pai AA, Pique-Regi R, et al. DNase I sensitivity QTLs are a major determinant of human expression variation. *Nature* 482:390–394 (2012). [doi:10.1038/nature10808](https://doi.org/10.1038/nature10808)
3. Pampari A, Shcherbina A, Kvon E, Kosicki M, et al. ChromBPNet: bias factorized, base-resolution deep learning models of chromatin accessibility reveal cis-regulatory sequence syntax, transcription factor footprints and regulatory variants. *bioRxiv* (2024). [doi:10.1101/2024.12.25.630221](https://doi.org/10.1101/2024.12.25.630221)
4. Lee D. LS-GKM: a new gkm-SVM for large-scale datasets. *Bioinformatics* 32(14):2196–2198 (2016). [doi:10.1093/bioinformatics/btw142](https://doi.org/10.1093/bioinformatics/btw142)
5. Ghandi M, Lee D, Mohammad-Noori M, Beer MA. Enhanced regulatory sequence prediction using gapped k-mer features. *PLoS Comput Biol* 10(7):e1003711 (2014). [doi:10.1371/journal.pcbi.1003711](https://doi.org/10.1371/journal.pcbi.1003711)